Connect to drive

In [1]:
cd

/root


In [2]:
ls

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Git clone the NCRFPP git repo

In [4]:
import os
if not os.path.exists('NCRFpp'):
    !git clone https://github.com/jiesutd/NCRFpp.git
else:
    print('Repository already exists.')

Cloning into 'NCRFpp'...
remote: Enumerating objects: 768, done.
remote: Total 768 (delta 0), reused 0 (delta 0), pack-reused 768 (from 1)
Receiving objects: 100% (768/768), 6.89 MiB | 11.27 MiB/s, done.
Resolving deltas: 100% (484/484), done.


In [5]:
ls

NCRFpp/


Add requirements.py and install the required libraries

In [6]:
with open('requirements.txt', 'w') as f:
    f.write('torch\nnumpy\n')
!pip install -r requirements.txt

Unzip the data file into project directory

In [7]:
!unzip -o "/content/drive/My Drive/MyAgriNER/data.zip" -d /NCRFpp

Archive:  /content/drive/My Drive/MyAgriNER/data.zip
   creating: /NCRFpp/data/
  inflating: /NCRFpp/data/first_sem_agri_bio_syllable.conll  
  inflating: /NCRFpp/data/first_sem_agri_bio_word.conll  
  inflating: /NCRFpp/data/first_sem_agri_bioes_syllable.conll  
  inflating: /NCRFpp/data/first_sem_agri_bioes_word.conll  
  inflating: /NCRFpp/data/burmese_agri_syllable.emb  
  inflating: /NCRFpp/data/burmese_agri_word.emb  


check if the data files are there

In [9]:
ls /NCRFpp/data

burmese_agri_syllable.emb            first_sem_agri_bioes_word.conll
burmese_agri_word.emb                first_sem_agri_bio_syllable.conll
first_sem_agri_bioes_syllable.conll  first_sem_agri_bio_word.conll


Slplit files into folder with train, dev and test

In [22]:
import os
import random
import glob

def split_conll_file(file_path, train_ratio=0.8, dev_ratio=0.1):
    if not os.path.exists(file_path):
        print(f"File {file_path} not found.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    # CoNLL files usually separate sentences/sequences with empty lines
    sequences = content.split('\n\n')
    random.seed(42)  # For reproducibility
    random.shuffle(sequences)

    total = len(sequences)
    train_end = int(total * train_ratio)
    dev_end = train_end + int(total * dev_ratio)

    train_data = sequences[:train_end]
    dev_data = sequences[train_end:dev_end]
    test_data = sequences[dev_end:]

    # Create a subfolder based on the filename
    file_name = os.path.basename(file_path)
    folder_name = file_name.replace('.conll', '')
    target_dir = os.path.join(os.path.dirname(file_path), folder_name)
    os.makedirs(target_dir, exist_ok=True)

    splits = {
        f'train.{file_name}': train_data,
        f'dev.{file_name}': dev_data,
        f'test.{file_name}': test_data
    }

    for out_name, data in splits.items():
        out_path = os.path.join(target_dir, out_name)
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write('\n\n'.join(data) + '\n')
        print(f"Saved {len(data)} sequences to {out_path}")

# Dynamically process all .conll files in the directory that are not already splits
data_dir = '/NCRFpp/data'
conll_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir)
               if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]

for file in conll_files:
    split_conll_file(file)

Saved 9476 sequences to /NCRFpp/data/first_sem_agri_bio_syllable/train.first_sem_agri_bio_syllable.conll
Saved 1184 sequences to /NCRFpp/data/first_sem_agri_bio_syllable/dev.first_sem_agri_bio_syllable.conll
Saved 1185 sequences to /NCRFpp/data/first_sem_agri_bio_syllable/test.first_sem_agri_bio_syllable.conll
Saved 9476 sequences to /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
Saved 1184 sequences to /NCRFpp/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
Saved 1185 sequences to /NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bioes_word.conll
Saved 9476 sequences to /NCRFpp/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
Saved 1184 sequences to /NCRFpp/data/first_sem_agri_bio_word/dev.first_sem_agri_bio_word.conll
Saved 1185 sequences to /NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
Saved 9476 sequences to /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllab

In [23]:
# rm /NCRFpp/*.config

rm: cannot remove '/NCRFpp/*.config': No such file or directory


In [33]:
import os
import glob

def create_config(name, train_path, dev_path, test_path):
    config_content = f"""
### use # to comment out the configure item

### I/O ###
train_dir=/NCRFpp/{train_path}
dev_dir=/NCRFpp/{dev_path}
test_dir=/NCRFpp/{test_path}
model_dir=/NCRFpp/models/{name}
# word_emb_dir=data/sample.word.emb

#raw_dir=
#decode_dir=
#dset_dir=
#load_model_dir=
#char_emb_dir=

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=50
char_emb_dim=30

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN
#feature=[POS] emb_size=20
#feature=[Cap] emb_size=20
#nbest=1

###TrainingSetting###
status=train
optimizer=SGD
iteration=1
batch_size=10
ave_batch_loss=False

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim=200
dropout=0.5
lstm_layer=1
bilstm=True
learning_rate=0.015
lr_decay=0.05
momentum=0
l2=1e-8
gpu
#clip=
"""
    # Save config in the root NCRFpp directory
    config_path = f"/NCRFpp/{name}.train.config"
    with open(config_path, 'w') as f:
        f.write(config_content.strip())
    print(f"Created config: {config_path}")

# Dynamic detection of files in data directory
data_dir = '/NCRFpp/data'
os.makedirs('/NCRFpp/models', exist_ok=True)

# Find all original conll names (e.g., bio.conll, bio_syl.conll)
conll_files = [f for f in os.listdir(data_dir) if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]

for base_file in conll_files:
    name = base_file.replace('.conll', '')

    # Construct paths relative to the execution context (NCRFpp folder)
    # Now pointing to the subfolders created by the splitter
    train_p = f"data/{name}/train.{base_file}"
    dev_p = f"data/{name}/dev.{base_file}"
    test_p = f"data/{name}/test.{base_file}"

    # Verify files exist before creating config
    if all(os.path.exists(os.path.join('/NCRFpp', p)) for p in [train_p, dev_p, test_p]):
        create_config(name, train_p, dev_p, test_p)
    else:
        print(f"Skipping {name}: Missing split files in subfolder {name}.")

Created config: /NCRFpp/first_sem_agri_bio_syllable.train.config
Created config: /NCRFpp/first_sem_agri_bioes_word.train.config
Created config: /NCRFpp/first_sem_agri_bio_word.train.config
Created config: /NCRFpp/first_sem_agri_bioes_syllable.train.config


In [34]:
import os
import glob

def create_config(name, train_path, dev_path, test_path):
    word_emb_dir_setting = ""
    if "word" in name:
        word_emb_dir_setting = "word_emb_dir=/NCRFpp/data/burmese_agri_word.emb"
    elif "syllable" in name:
        word_emb_dir_setting = "word_emb_dir=/NCRFpp/data/burmese_agri_syllable.emb"

    config_content = f"""
### use # to comment out the configure item

### I/O ###
train_dir=/NCRFpp/{train_path}
dev_dir=/NCRFpp/{dev_path}
test_dir=/NCRFpp/{test_path}
model_dir=/NCRFpp/models/with_emb.{name}
{word_emb_dir_setting}

#raw_dir=
#decode_dir=
#dset_dir=
#load_model_dir=
#char_emb_dir=

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=50
char_emb_dim=30

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN
#feature=[POS] emb_size=20
#feature=[Cap] emb_size=20
#nbest=1

###TrainingSetting###
status=train
optimizer=SGD
iteration=1
batch_size=10
ave_batch_loss=False

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim=200
dropout=0.5
lstm_layer=1
bilstm=True
learning_rate=0.015
lr_decay=0.05
momentum=0
l2=1e-8
gpu
#clip=
"""
    # Save config in the root NCRFpp directory
    config_path = f"/NCRFpp/{name}.with_emb.train.config"
    with open(config_path, 'w') as f:
        f.write(config_content.strip())
    print(f"Created config: {config_path}")

# Dynamic detection of files in data directory
data_dir = '/NCRFpp/data'
os.makedirs('/NCRFpp/models', exist_ok=True)

# Find all original conll names (e.g., bio.conll, bio_syl.conll)
conll_files = [f for f in os.listdir(data_dir) if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]

for base_file in conll_files:
    name = base_file.replace('.conll', '')

    # Construct paths relative to the execution context (NCRFpp folder)
    # Now pointing to the subfolders created by the splitter
    train_p = f"data/{name}/train.{base_file}"
    dev_p = f"data/{name}/dev.{base_file}"
    test_p = f"data/{name}/test.{base_file}"

    # Verify files exist before creating config
    if all(os.path.exists(os.path.join('/NCRFpp', p)) for p in [train_p, dev_p, test_p]):
        create_config(name, train_p, dev_p, test_p)
    else:
        print(f"Skipping {name}: Missing split files in subfolder {name}.")

Created config: /NCRFpp/first_sem_agri_bio_syllable.with_emb.train.config
Created config: /NCRFpp/first_sem_agri_bioes_word.with_emb.train.config
Created config: /NCRFpp/first_sem_agri_bio_word.with_emb.train.config
Created config: /NCRFpp/first_sem_agri_bioes_syllable.with_emb.train.config


In [35]:
cat /NCRFpp/first_sem_agri_bioes_word.with_emb.train.config

### use # to comment out the configure item

### I/O ###
train_dir=/NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
dev_dir=/NCRFpp/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
test_dir=/NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bioes_word.conll
model_dir=/NCRFpp/models/with_emb.first_sem_agri_bioes_word
word_emb_dir=/NCRFpp/data/burmese_agri_word.emb

#raw_dir=
#decode_dir=
#dset_dir=
#load_model_dir=
#char_emb_dir=

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=50
char_emb_dim=30

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN
#feature=[POS] emb_size=20
#feature=[Cap] emb_size=20
#nbest=1

###TrainingSetting###
status=train
optimizer=SGD
iteration=1
batch_size=10
ave_batch_loss=False

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim=200
dropout=0.5
lstm_layer=1
bilstm=True
learning_rate=0.015
lr_decay=0.05
momentum=0

In [36]:
ls /NCRFpp/

data/
first_sem_agri_bioes_syllable.train.config
first_sem_agri_bioes_syllable.with_emb.train.config
first_sem_agri_bioes_word.train.config
first_sem_agri_bioes_word.with_emb.train.config
first_sem_agri_bio_syllable.train.config
first_sem_agri_bio_syllable.with_emb.train.config
first_sem_agri_bio_word.train.config
first_sem_agri_bio_word.with_emb.train.config
models/


In [37]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_word.with_emb.train.config

Seed num: 42
MODEL: train
Load pretrained word embedding, norm: False, dir: /NCRFpp/data/burmese_agri_word.emb
Embedding:
     pretrain word:60670, prefect match:10468, case_match:0, oov:4450, oov%:0.29827736443461356
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 139
     Word embedding  dir: /NCRFpp/data/burmese_agri_word.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
     Dev    file directory: /

In [66]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_word.train.config

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 139
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bioes_word.conll
     Raw    file directory: None
     Dset   

In [70]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_syllable.with_emb.train.config

Seed num: 42
MODEL: train
Load pretrained word embedding, norm: False, dir: /NCRFpp/data/burmese_agri_syllable.emb
Embedding:
     pretrain word:17187, prefect match:2217, case_match:0, oov:237, oov%:0.09653767820773931
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 132
     Word embedding  dir: /NCRFpp/data/burmese_agri_syllable.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
     Dev    file

In [71]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_syllable.train.config

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 132
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/dev.first_sem_agri_bioes_syllable.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/test.first_sem_agri_bioes_syllable.conll
     Raw    file direc

In [72]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bio_word.with_emb.train.config

Seed num: 42
MODEL: train
Load pretrained word embedding, norm: False, dir: /NCRFpp/data/burmese_agri_word.emb
Embedding:
     pretrain word:60670, prefect match:10468, case_match:0, oov:4450, oov%:0.29827736443461356
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: /NCRFpp/data/burmese_agri_word.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
     Dev    file directory: /NCRFpp

In [73]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bio_word.train.config

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_word/dev.first_sem_agri_bio_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
     Raw    file directory: None
     Dset   file directory

In [75]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bio_syllable.with_emb.train.config

Seed num: 42
MODEL: train
Load pretrained word embedding, norm: False, dir: /NCRFpp/data/burmese_agri_syllable.emb
Embedding:
     pretrain word:17187, prefect match:2217, case_match:0, oov:237, oov%:0.09653767820773931
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: /NCRFpp/data/burmese_agri_syllable.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_syllable/train.first_sem_agri_bio_syllable.conll
     Dev    file direc

In [74]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bio_syllable.train.config

Seed num: 42
MODEL: train
Training model...
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_syllable/train.first_sem_agri_bio_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_syllable/dev.first_sem_agri_bio_syllable.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bio_syllable/test.first_sem_agri_bio_syllable.conll
     Raw    file directory: None
   

### Decoding Configuration Generator
This script generates `.decode.config` files for testing. It assumes that training has completed and produced model files in the `/NCRFpp/models/` directory.

In [76]:
import os
import glob

def create_decode_config(name, test_path, model_path, dset_path):
    # Ensure output directory exists
    os.makedirs('/NCRFpp/output', exist_ok=True)

    decode_content = f"""
### I/O ###
status=decode
raw_dir=/NCRFpp/{test_path}
decode_dir=/NCRFpp/output/{name}.test.out
dset_dir={dset_path}
load_model_dir={model_path}

nbest=10
#gpu
"""
    config_path = f"/NCRFpp/{name}.decode.config"
    with open(config_path, 'w') as f:
        f.write(decode_content.strip())
    print(f"Created decode config: {config_path}")

# Updated detection logic for models in the root models folder
models_root = '/NCRFpp/models'
if os.path.exists(models_root):
    # Find all .dset files to identify trained datasets
    dset_paths = glob.glob(os.path.join(models_root, "*.dset"))

    for dset_path in dset_paths:
        # e.g., /NCRFpp/models/bio.dset -> name = 'bio'
        name = os.path.basename(dset_path).replace('.dset', '')

        # Find matching models for this name (e.g., bio.0.model, bio.1.model)
        model_files = sorted(glob.glob(os.path.join(models_root, f"{name}.*.model")))

        if model_files:
            # Use the latest model and the standard test path
            test_p = f"data/{name.replace('with_emb.', '')}/test.{name.replace('with_emb.', '')}.conll"
            create_decode_config(name, test_p, model_files[-1], dset_path)
        else:
            print(f"No models found for {name} in {models_root}")
else:
    print(f"Models directory {models_root} does not exist.")

Created decode config: /NCRFpp/with_emb.first_sem_agri_bioes_word.decode.config
Created decode config: /NCRFpp/first_sem_agri_bioes_word.decode.config
Created decode config: /NCRFpp/with_emb.first_sem_agri_bio_word.decode.config
Created decode config: /NCRFpp/with_emb.first_sem_agri_bio_syllable.decode.config
Created decode config: /NCRFpp/first_sem_agri_bioes_syllable.decode.config
Created decode config: /NCRFpp/first_sem_agri_bio_syllable.decode.config
Created decode config: /NCRFpp/with_emb.first_sem_agri_bioes_syllable.decode.config
Created decode config: /NCRFpp/first_sem_agri_bio_word.decode.config


In [77]:
ls /NCRFpp

data/
first_sem_agri_bioes_syllable.decode.config
first_sem_agri_bioes_syllable.train.config
first_sem_agri_bioes_syllable.with_emb.train.config
first_sem_agri_bioes_word.decode.config
first_sem_agri_bioes_word.train.config
first_sem_agri_bioes_word.with_emb.train.config
first_sem_agri_bio_syllable.decode.config
first_sem_agri_bio_syllable.train.config
first_sem_agri_bio_syllable.with_emb.train.config
first_sem_agri_bio_word.decode.config
first_sem_agri_bio_word.train.config
first_sem_agri_bio_word.with_emb.train.config
models/
output/
with_emb.first_sem_agri_bioes_syllable.decode.config
with_emb.first_sem_agri_bioes_word.decode.config
with_emb.first_sem_agri_bio_syllable.decode.config
with_emb.first_sem_agri_bio_word.decode.config


In [48]:
# rm /NCRFpp/demo.with_emb.first_sem_agri_bioes_word.decode.config

In [78]:
ls /NCRFpp/models

first_sem_agri_bioes_syllable.0.model
first_sem_agri_bioes_syllable.dset
first_sem_agri_bioes_word.0.model
first_sem_agri_bioes_word.dset
first_sem_agri_bio_syllable.0.model
first_sem_agri_bio_syllable.dset
first_sem_agri_bio_word.0.model
first_sem_agri_bio_word.dset
with_emb.first_sem_agri_bioes_syllable.0.model
with_emb.first_sem_agri_bioes_syllable.dset
with_emb.first_sem_agri_bioes_word.0.model
with_emb.first_sem_agri_bioes_word.dset
with_emb.first_sem_agri_bio_syllable.0.model
with_emb.first_sem_agri_bio_syllable.dset
with_emb.first_sem_agri_bio_word.0.model
with_emb.first_sem_agri_bio_word.dset


In [79]:
ls /NCRFpp/output

first_sem_agri_bioes_word.test.out  with_emb.first_sem_agri_bioes_word.test.out


In [63]:
!python NCRFpp/main.py --config /NCRFpp/with_emb.first_sem_agri_bioes_word.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bioes_word.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 139
     Word embedding  dir: /NCRFpp/data/burmese_agri_word.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bio

In [69]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_word.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bioes_word.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 139
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_word/train.first_sem_agri_bioes_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_word/dev.first_sem_agri_bioes_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bioes_word/test.first_sem_agri_bio

In [80]:
!python NCRFpp/main.py --config /NCRFpp/with_emb.first_sem_agri_bioes_syllable.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bioes_syllable/test.first_sem_agri_bioes_syllable.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 132
     Word embedding  dir: /NCRFpp/data/burmese_agri_syllable.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/dev.first_sem_agri_bioes_syllable.conll
     Test   file directory: /NCR

In [81]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bioes_syllable.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bioes_syllable/test.first_sem_agri_bioes_syllable.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BMES
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 132
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/train.first_sem_agri_bioes_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bioes_syllable/dev.first_sem_agri_bioes_syllable.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bioes_sylla

In [85]:
!python NCRFpp/main.py --config /NCRFpp/with_emb.first_sem_agri_bio_word.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: /NCRFpp/data/burmese_agri_word.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_word/dev.first_sem_agri_bio_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bio_word/test.fir

In [82]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bio_word.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 14919
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_word/train.first_sem_agri_bio_word.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_word/dev.first_sem_agri_bio_word.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bio_word/test.first_sem_agri_bio_word.conll
    

In [83]:
!python NCRFpp/main.py --config /NCRFpp/with_emb.first_sem_agri_bio_syllable.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bio_syllable/test.first_sem_agri_bio_syllable.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: /NCRFpp/data/burmese_agri_syllable.emb
     Char embedding  dir: None
     Word embedding size: 200
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_syllable/train.first_sem_agri_bio_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_syllable/dev.first_sem_agri_bio_syllable.conll
     Test   file directory: /NCRFpp/data/first

In [84]:
!python NCRFpp/main.py --config /NCRFpp/first_sem_agri_bio_syllable.decode.config

Seed num: 42
MODEL: decode
/NCRFpp/data/first_sem_agri_bio_syllable/test.first_sem_agri_bio_syllable.conll
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DATA SUMMARY START:
 I/O:
     Start   Sequence   Laebling   task...
     Tag          scheme: BIO
     Split         token:  ||| 
     MAX SENTENCE LENGTH: 250
     MAX   WORD   LENGTH: -1
     Number   normalized: True
     Word  alphabet size: 2455
     Char  alphabet size: 109
     Label alphabet size: 72
     Word embedding  dir: None
     Char embedding  dir: None
     Word embedding size: 50
     Char embedding size: 30
     Norm   word     emb: False
     Norm   char     emb: False
     Train  file directory: /NCRFpp/data/first_sem_agri_bio_syllable/train.first_sem_agri_bio_syllable.conll
     Dev    file directory: /NCRFpp/data/first_sem_agri_bio_syllable/dev.first_sem_agri_bio_syllable.conll
     Test   file directory: /NCRFpp/data/first_sem_agri_bio_syllable/test.first_s

In [86]:
ls /NCRFpp/output/

first_sem_agri_bioes_syllable.test.out
first_sem_agri_bioes_word.test.out
first_sem_agri_bio_syllable.test.out
first_sem_agri_bio_word.test.out
with_emb.first_sem_agri_bioes_syllable.test.out
with_emb.first_sem_agri_bioes_word.test.out
with_emb.first_sem_agri_bio_syllable.test.out
with_emb.first_sem_agri_bio_word.test.out


In [65]:
!head /NCRFpp/output/with_emb.first_sem_agri_bioes_word.test.out -n 200

# 0.3300 0.2313 0.1856 0.1301 0.0283 0.0275 0.0204 0.0199 0.0155 0.0115
သို့ O O O O O O O O O O
ဖြစ် O O O O O O O O O O
၍ O O O O O O O O O O
စပါး B-CROP S-CROP B-CROP S-CROP B-CROP S-CROP S-CROP S-CROP S-CROP S-CROP
ပင် E-CROP O E-CROP O E-CROP E-CROP E-SEED O E-CROP E-SEED
များ O O O O O O O O O O
၏ O O O O O O O O O O
အမြစ် S-CROP_PART S-CROP_PART S-CROP_PART S-CROP_PART S-CROP_PART S-CROP_PART S-CROP_PART S-CROP_PART S-CROP_PART S-CROP_PART
ဇုန်ဝန်းကျင် O O O O O O O O O O
သို့ O O O O O O O O O O
ရေ O O O O O O O O O O
ကို O O O O O O O O O O
စိုစွတ် S-WEATHER S-WEATHER O O B-WEATHER S-WEATHER S-WEATHER B-WEATHER O O
ရုံ O O O O O O O O O O
သာ O O O O O O O O O O
ပေးသွင်းရုံ O O O O O O O O O O
ဖြင့် O O O O O O O O O O
စိုက်ပျိုး S-FARM_OP S-FARM_OP S-FARM_OP S-FARM_OP S-FARM_OP S-FARM_OP S-FARM_OP S-FARM_OP S-FARM_OP S-FARM_OP
အောင်မြင် O O O O O O O O O O
နိုင် O O O O O O O O O O
ခြင်း O O O O O O O O O O
ဖြစ် O O O O O O O O O O
သည် O O O O O O O O O O
။ O O O O O O O O O O

In [87]:
import shutil
from datetime import datetime

# Define the destination path in Google Drive
drive_export_path = '/content/drive/My Drive/MyAgriNER/first_sem_my_agri_export'
os.makedirs(drive_export_path, exist_ok=True)

# Source directories to backup
sources = {
    'models': '/NCRFpp/models',
    'output': '/NCRFpp/output',
    'configs': '/NCRFpp/*.config'
}

print(f"Starting backup to {drive_export_path}...")

# Copy models and output folders
for folder in ['models', 'output']:
    src = f'/NCRFpp/{folder}'
    dst = os.path.join(drive_export_path, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✓ Copied {folder} to Drive.")

# Copy config files specifically
import glob
for config_file in glob.glob('/NCRFpp/*.config'):
    shutil.copy(config_file, drive_export_path)

print("\nBackup complete! Your models, outputs, and configs are now safe in your Google Drive.")

Starting backup to /content/drive/My Drive/MyAgriNER/first_sem_my_agri_export...
✓ Copied models to Drive.
✓ Copied output to Drive.

Backup complete! Your models, outputs, and configs are now safe in your Google Drive.
